# Audio Transformer Encoder for FMA Small

This notebook builds an encoder-only transformer genre classifier for `fma_small` with PyTorch.

Pipeline:
1. Load FMA metadata and audio paths.
2. Use a manageable balanced split from `fma_small`.
3. Resample clips to one sample rate.
4. Trim or pad every clip to a fixed duration.
5. Convert audio to log-mel spectrograms.
6. Split spectrograms into patches and feed them to a transformer encoder.
7. Train with cross-entropy on genre labels.

## 1. Setup

If the imports below fail in your Jupyter kernel, run this once in a notebook cell, restart the kernel, then continue:

```python
%pip install torch torchaudio pandas numpy tqdm
```

In [44]:
import os
from contextlib import nullcontext
from pathlib import Path
import random

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchaudio

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x

# Fix the random seeds so that the sampled subset and training behavior are repeatable.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Resolve the dataset paths relative to either the current working directory or its parent.
PROJECT_CANDIDATES = [Path.cwd(), Path.cwd().parent]

for candidate in PROJECT_CANDIDATES:
    audio_dir = candidate / "fma_small"
    metadata_path = candidate / "fma_metadata" / "tracks.csv"
    if audio_dir.exists() and metadata_path.exists():
        PROJECT_DIR = candidate.resolve()
        AUDIO_DIR = audio_dir.resolve()
        METADATA_PATH = metadata_path.resolve()
        break
else:
    raise FileNotFoundError(
        "Could not find fma_small/ and fma_metadata/tracks.csv from the current directory or its parent."
    )

# Use the fastest available device: NVIDIA GPU, Apple Silicon GPU, or CPU.
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
device
PROJECT_DIR

PosixPath('/home/lilyd/acml-project')

## 2. Load FMA Small Metadata

In [45]:
# FMA stores audio as fma_small/000/000002.mp3, so this rebuilds the path from the track id.
def track_id_to_audio_path(track_id: int) -> Path:
    track_id = int(track_id)
    filename = f"{track_id:06d}.mp3"
    return AUDIO_DIR / filename[:3] / filename


# Cache the audio validation results so we do not fully scan the MP3s every notebook run.
AUDIO_VALIDATION_CACHE = PROJECT_DIR / "fma_small_audio_validation.csv"


def can_decode_audio(path: Path) -> tuple[bool, str]:
    try:
        audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)
        if sample_rate <= 0 or audio.size == 0:
            return False, "empty_audio"
        return True, ""
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"


def filter_decodable_audio(frame: pd.DataFrame) -> pd.DataFrame:
    paths = frame["audio_path"].map(str)
    validation = None

    if AUDIO_VALIDATION_CACHE.exists():
        cached = pd.read_csv(AUDIO_VALIDATION_CACHE)
        if {"audio_path", "is_valid", "error"}.issubset(cached.columns):
            cached = cached.drop_duplicates(subset=["audio_path"], keep="last")
            if set(paths).issubset(set(cached["audio_path"])):
                validation = cached.set_index("audio_path")

    if validation is None:
        rows = []
        for path in tqdm(frame["audio_path"], desc="Validating audio files"):
            is_valid, error = can_decode_audio(path)
            rows.append({
                "audio_path": str(path),
                "is_valid": is_valid,
                "error": error,
            })
        validation = pd.DataFrame(rows).drop_duplicates(subset=["audio_path"], keep="last")
        validation.to_csv(AUDIO_VALIDATION_CACHE, index=False)
        validation = validation.set_index("audio_path")

    filtered = frame.copy()
    filtered["audio_path_str"] = filtered["audio_path"].map(str)
    filtered = filtered.join(validation, on="audio_path_str")
    bad_rows = filtered[~filtered["is_valid"].fillna(False)].copy()
    if not bad_rows.empty:
        print(f"Dropping {len(bad_rows):,} corrupt/unreadable audio files before training.")
        display(bad_rows[["track_id", "split", "genre", "audio_path", "error"]].head(10))

    return filtered[filtered["is_valid"].fillna(False)].drop(columns=["audio_path_str", "is_valid", "error"]).copy()


def load_fma_small_metadata(validate_audio: bool = True) -> pd.DataFrame:
    # tracks.csv has a two-row header, which creates pandas MultiIndex columns like ("track", "genre_top").
    tracks = pd.read_csv(METADATA_PATH, index_col=0, header=[0, 1])

    # Keep only the columns needed for supervised genre classification.
    frame = pd.DataFrame({
        "track_id": tracks.index.astype(int),
        "subset": tracks[("set", "subset")],
        "split": tracks[("set", "split")],
        "genre": tracks[("track", "genre_top")],
        "duration": tracks[("track", "duration")],
    })

    # Attach each metadata row to its local MP3 file and remove missing/unlabelled tracks.
    frame["audio_path"] = frame["track_id"].apply(track_id_to_audio_path)
    frame = frame[
        (frame["subset"] == "small")
        & frame["genre"].notna()
        & frame["audio_path"].map(lambda path: path.exists())
    ].copy()

    if validate_audio:
        frame = filter_decodable_audio(frame)

    return frame


metadata = load_fma_small_metadata(validate_audio=True)
metadata.head()


Dropping 6 corrupt/unreadable audio files before training.


,track_id,split,genre,audio_path,error
track_id,,,,,
98565,98565,training,Hip-Hop,/home/lilyd/acml-project/fma_small/098/098565.mp3,LibsndfileError: Unspecified internal error.
98567,98567,training,Hip-Hop,/home/lilyd/acml-project/fma_small/098/098567.mp3,LibsndfileError: Unspecified internal error.
98569,98569,training,Hip-Hop,/home/lilyd/acml-project/fma_small/098/098569.mp3,LibsndfileError: Unspecified internal error.
99134,99134,training,Electronic,/home/lilyd/acml-project/fma_small/099/099134.mp3,LibsndfileError: Error opening '/home/lilyd/ac...
108925,108925,training,Rock,/home/lilyd/acml-project/fma_small/108/108925.mp3,LibsndfileError: Error opening '/home/lilyd/ac...
133297,133297,training,Experimental,/home/lilyd/acml-project/fma_small/133/133297.mp3,LibsndfileError: Error opening '/home/lilyd/ac...


,track_id,subset,split,genre,duration,audio_path
track_id,,,,,,
2,2,small,training,Hip-Hop,168,/home/lilyd/acml-project/fma_small/000/000002.mp3
5,5,small,training,Hip-Hop,206,/home/lilyd/acml-project/fma_small/000/000005.mp3
10,10,small,training,Pop,161,/home/lilyd/acml-project/fma_small/000/000010.mp3
140,140,small,training,Folk,253,/home/lilyd/acml-project/fma_small/000/000140.mp3
141,141,small,training,Folk,182,/home/lilyd/acml-project/fma_small/000/000141.mp3


In [46]:
print(f"Usable FMA small clips: {len(metadata):,}")
print(metadata["split"].value_counts())
print(metadata["genre"].value_counts().sort_index())

Usable FMA small clips: 7,994
split
training      6394
validation     800
test           800
Name: count, dtype: int64
genre
Electronic        999
Experimental      999
Folk             1000
Hip-Hop           997
Instrumental     1000
International    1000
Pop              1000
Rock              999
Name: count, dtype: int64


## 3. Pick a Manageable Balanced Split

`fma_small` has 8000 tracks. The defaults below keep training quick while preserving the official FMA train/validation/test split and balancing by genre. Set both limits to `None` to use all available FMA small clips.

In [47]:
# These caps make training manageable on a laptop while keeping the classes balanced.
# Increase them, or set them to None, when you want to train on more of fma_small.
MAX_TRAIN_PER_GENRE = None
MAX_EVAL_PER_GENRE = None


required_metadata_columns = {"track_id", "subset", "split", "genre", "audio_path"}
if not required_metadata_columns.issubset(metadata.columns):
    # If a previous cell left metadata as the raw tracks.csv table, rebuild the compact table used below.
    metadata = load_fma_small_metadata()

missing_columns = required_metadata_columns.difference(metadata.columns)
if missing_columns:
    raise ValueError(f"metadata is missing required columns: {sorted(missing_columns)}")


def cap_per_genre(frame: pd.DataFrame, max_per_genre: int | None) -> pd.DataFrame:
    # Sample up to max_per_genre clips from each genre so common genres do not dominate training.
    if frame.empty:
        return frame.reset_index(drop=True)
    if max_per_genre is None:
        return frame.sample(frac=1, random_state=SEED).reset_index(drop=True)

    sampled_groups = [
        group.sample(min(len(group), max_per_genre), random_state=SEED)
        for _, group in frame.groupby("genre", sort=False)
    ]
    return (
        pd.concat(sampled_groups)
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=True)
    )


# Preserve the official FMA split instead of randomly mixing train/validation/test tracks.
train_df = cap_per_genre(metadata[metadata["split"] == "training"], MAX_TRAIN_PER_GENRE)
val_df = cap_per_genre(metadata[metadata["split"] == "validation"], MAX_EVAL_PER_GENRE)
test_df = cap_per_genre(metadata[metadata["split"] == "test"], MAX_EVAL_PER_GENRE)

# Convert text genres to integer class ids for CrossEntropyLoss.
genres = sorted(train_df["genre"].unique())
genre_to_idx = {genre: idx for idx, genre in enumerate(genres)}
idx_to_genre = {idx: genre for genre, idx in genre_to_idx.items()}

for frame in (train_df, val_df, test_df):
    frame["label"] = frame["genre"].map(genre_to_idx).astype(int)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(genre_to_idx)


Train: 6,394 | Val: 800 | Test: 800
{'Electronic': 0, 'Experimental': 1, 'Folk': 2, 'Hip-Hop': 3, 'Instrumental': 4, 'International': 5, 'Pop': 6, 'Rock': 7}


## 4. Dataset: Resample, Trim/Pad, Log-Mel Spectrogram

In [48]:
# All clips are converted to the same sample rate and same length before feature extraction.
SAMPLE_RATE = 16_000
CLIP_SECONDS = 10
NUM_SAMPLES = SAMPLE_RATE * CLIP_SECONDS

# Spectrogram settings: about 25 ms windows and 10 ms hop at 16 kHz.
N_FFT = 400
HOP_LENGTH = 160
# 64 mel bins avoids empty filters with a 400-point FFT while keeping features compact.
N_MELS = 64


class FMASmallDataset(Dataset):
    # This Dataset keeps CPU-side work to audio decode, resample, crop, and pad.
    # Spectrogram conversion happens on the GPU later so training can use it more.
    def __init__(self, frame: pd.DataFrame, train: bool = False):
        self.frame = frame.reset_index(drop=True)
        self.train = train

    def __len__(self) -> int:
        return len(self.frame)

    def _load_audio(self, path: Path) -> torch.Tensor:
        # soundfile avoids torchaudio.load's TorchCodec/FFmpeg shared-library dependency on this setup.
        try:
            audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)
        except Exception as exc:
            raise RuntimeError(f"Failed to decode audio file: {path}") from exc
        waveform = torch.from_numpy(audio.T)
        # Average stereo channels into mono so every example has one input channel.
        waveform = waveform.mean(dim=0, keepdim=True)

        # Resample every file to the same sample rate before trimming or padding.
        if sample_rate != SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sample_rate, SAMPLE_RATE)

        # Long clips are cropped to a fixed duration. Training uses a random crop for light augmentation.
        if waveform.shape[1] > NUM_SAMPLES:
            if self.train:
                start = random.randint(0, waveform.shape[1] - NUM_SAMPLES)
            else:
                start = (waveform.shape[1] - NUM_SAMPLES) // 2
            waveform = waveform[:, start:start + NUM_SAMPLES]
        # Short clips are padded with silence at the end so batching works cleanly.
        elif waveform.shape[1] < NUM_SAMPLES:
            pad_amount = NUM_SAMPLES - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_amount))

        return waveform

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        waveform = self._load_audio(row["audio_path"])
        label = torch.tensor(row["label"], dtype=torch.long)
        return waveform, label


train_dataset = FMASmallDataset(train_df, train=True)
val_dataset = FMASmallDataset(val_df)
test_dataset = FMASmallDataset(test_df)

example_waveform, example_label = train_dataset[0]
example_waveform.shape, idx_to_genre[int(example_label)]

(torch.Size([1, 160000]), 'Electronic')

In [49]:
# Lower this if your machine runs out of memory; raise it if training is stable and you have headroom.
BATCH_SIZE = 32

# Audio decoding and spectrogram extraction happen in the DataLoader workers, so using multiple
# workers helps keep the device fed instead of doing all preprocessing on the main training thread.
CPU_COUNT = os.cpu_count() or 1
NUM_WORKERS = 10
PIN_MEMORY = device == "cuda"

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
}

if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = 2

# Shuffle only the training loader; validation/test should stay deterministic.
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print(f"Device: {device} | CPU cores: {CPU_COUNT} | DataLoader workers: {NUM_WORKERS}")
batch_waveforms, batch_labels = next(iter(train_loader))
batch_waveforms.shape, batch_labels.shape

Device: cuda | CPU cores: 12 | DataLoader workers: 10


[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (3328) too large for available bit count (3240)


(torch.Size([32, 1, 160000]), torch.Size([32]))

## 5. Encoder-Only Transformer Model

The model uses a convolutional patch embedding over the log-mel spectrogram, prepends a learnable class token, then classifies the final class-token representation.

In [50]:
class AudioPreprocessor(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
        )
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")

    def forward(self, waveforms: torch.Tensor) -> torch.Tensor:
        specs = self.to_db(self.mel(waveforms))
        mean = specs.mean(dim=(-2, -1), keepdim=True)
        std = specs.std(dim=(-2, -1), keepdim=True)
        return (specs - mean) / (std + 1e-6)


preprocessor = AudioPreprocessor().to(device)
with torch.no_grad():
    example_spec = preprocessor(example_waveform.unsqueeze(0).to(device)).cpu().squeeze(0)


class SpectrogramTransformer(nn.Module):
    # Encoder-only transformer for spectrogram classification.
    def __init__(
        self,
        num_classes: int,
        input_shape: tuple[int, int, int],
        patch_size: tuple[int, int] = (16, 16),
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 4,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()
        channels, n_mels, n_frames = input_shape
        # A strided convolution cuts the spectrogram into non-overlapping patches and projects each patch.
        self.patch_embed = nn.Conv2d(channels, d_model, kernel_size=patch_size, stride=patch_size)

        # The transformer processes a sequence: one token per spectrogram patch plus one class token.
        num_patches = (n_mels // patch_size[0]) * (n_frames // patch_size[1])
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, d_model))
        self.pos_dropout = nn.Dropout(dropout)

        # Stacked self-attention layers let every patch attend to every other time-frequency patch.
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        # The classifier maps the final class token embedding to genre logits.
        self.head = nn.Linear(d_model, num_classes)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Input x has shape [batch, 1, mel_bins, time_frames].
        x = self.patch_embed(x)
        # Flatten the patch grid into a token sequence: [batch, patches, d_model].
        x = x.flatten(2).transpose(1, 2)
        # The class token summarizes the whole clip after self-attention.
        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = self.pos_dropout(x + self.pos_embed[:, :x.shape[1]])
        x = self.encoder(x)
        x = self.norm(x[:, 0])
        return self.head(x)


model = SpectrogramTransformer(
    num_classes=len(genres),
    input_shape=tuple(example_spec.shape),
).to(device)

sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

/tmp/ipykernel_79545/3427631323.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


596104

## 6. Train with Cross-Entropy

In [51]:
USE_AMP = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def run_epoch(model, loader, criterion, optimizer=None):
    # If an optimizer is supplied, this function trains; otherwise it evaluates.
    is_train = optimizer is not None
    model.train(is_train)
    preprocessor.train(is_train)

    total_loss = 0.0
    correct = 0
    total = 0

    # Disable gradient tracking during validation/test to save memory and time.
    with torch.set_grad_enabled(is_train):
        for waveforms, labels in tqdm(loader, leave=False):
            waveforms = waveforms.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            specs = preprocessor(waveforms)

            amp_context = (
                torch.autocast(device_type="cuda", dtype=torch.float16)
                if USE_AMP else nullcontext()
            )
            with amp_context:
                # Logits are raw class scores with shape [batch_size, num_genres].
                logits = model(specs)
                loss = criterion(logits, labels)

            if is_train:
                # Standard PyTorch training step: clear old gradients, backpropagate, then update weights.
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                # Gradient clipping helps keep transformer training stable.
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


# CrossEntropyLoss is the usual choice for single-label multi-class classification.
criterion = nn.CrossEntropyLoss()
# AdamW works well for transformers; weight decay provides a little regularization.
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

# Train for longer, but stop early if validation accuracy stalls.
EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
MIN_DELTA = 1e-4

# Match the cosine schedule to the total planned epochs.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_acc = 0.0
epochs_without_improvement = 0
best_model_path = PROJECT_DIR / "best_audio_transformer.pt"

history = []
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion)
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "lr": current_lr,
    })

    improved = val_acc > best_val_acc + MIN_DELTA

    # Save the checkpoint that performs best on validation data, not just the final epoch.
    if improved:
        best_val_acc = val_acc
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "genre_to_idx": genre_to_idx,
            "config": {
                "sample_rate": SAMPLE_RATE,
                "clip_seconds": CLIP_SECONDS,
                "n_fft": N_FFT,
                "hop_length": HOP_LENGTH,
                "n_mels": N_MELS,
                "input_shape": tuple(example_spec.shape),
            },
        }, best_model_path)
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_loss:.4f}, acc {train_acc:.3f} | "
        f"val loss {val_loss:.4f}, acc {val_acc:.3f} | "
        f"lr {current_lr:.2e}"
    )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping triggered after {epoch} epochs.")
        break

pd.DataFrame(history)

  0%|          | 0/200 [00:00<?, ?it/s]

[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (3328) too large for available bit count (3240)
[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (3360) too large for available bit count (3240)


  0%|          | 0/25 [00:00<?, ?it/s]

Epoch 01 | train loss 1.8198, acc 0.308 | val loss 1.7132, acc 0.329 | lr 2.99e-04


  0%|          | 0/200 [00:00<?, ?it/s]

...

  0%|          | 0/200 [00:00<?, ?it/s]

[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (3328) too large for available bit count (3240)
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (3360) too large for available bit count (3240)
[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!


  0%|          | 0/25 [00:00<?, ?it/s]

Epoch 10 | train loss 1.3694, acc 0.513 | val loss 1.5224, acc 0.456 | lr 2.25e-04
Early stopping triggered after 10 epochs.


,epoch,train_loss,train_acc,val_loss,val_acc,lr
0,1,1.819811,0.307945,1.713177,0.32875,0.000299
1,2,1.682892,0.377698,1.633614,0.40000,0.000297
2,3,1.605334,0.420551,1.636501,0.40625,0.000293
3,4,1.560364,0.434313,1.594848,0.40375,0.000287
4,5,1.516688,0.453237,1.525568,0.46875,0.000280
5,6,1.473268,0.471379,1.522149,0.41000,0.000271
6,7,1.453866,0.481389,1.547357,0.42625,0.000261
7,8,1.423507,0.495464,1.534691,0.43500,0.000250
8,9,1.393768,0.501720,1.495695,0.46250,0.000238
9,10,1.369440,0.512981,1.522361,0.45625,0.000225


## 7. Test Evaluation

In [52]:
# Reload the best validation checkpoint before measuring final test performance.
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_acc = run_epoch(model, test_loader, criterion)
print(f"Best validation accuracy: {best_val_acc:.3f}")
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.3f}")

  0%|          | 0/25 [00:00<?, ?it/s]

Best validation accuracy: 0.469
Test loss: 1.6289 | Test accuracy: 0.401


## 8. Inspect Predictions

In [55]:
# Look at a small batch of true labels and predicted labels for a quick qualitative check.
model.eval()
specs, labels = next(iter(test_loader))
with torch.no_grad():
    logits = model(specs.to(device))
    predictions = logits.argmax(dim=1).cpu()

pd.DataFrame({
    "true": [idx_to_genre[int(label)] for label in labels],
    "predicted": [idx_to_genre[int(prediction)] for prediction in predictions],
}).head(16)

RuntimeError: Given groups=1, weight of size [128, 1, 16, 16], expected input[1, 32, 1, 160000] to have 1 channels, but got 32 channels instead